In [73]:
import numpy as np
import optimize
import numpy as np
import pandas as pd
import yfinance as yf

In [85]:
conversion_to_annual = {
    "1d": 252,
    "1wk": 52,
    "1mo": 12,
}

Current portfolio

In [92]:
# My current portfolio
tickers = ["VWCE.DE", "IUSN.DE"] # current portfolio
interval = "1mo"  # "1d", "1wk", or "1mo"
anualization_factor = conversion_to_annual[interval]
annual_risk_free = 0.02
anualization_factor = conversion_to_annual[interval]
risk_free = (1 + annual_risk_free) ** (1 / anualization_factor) - 1

data = yf.download(
    tickers,
    interval=interval,
    auto_adjust=True,
    period="max",
)
timeseries = data["Close"]

returns = timeseries.pct_change()
mu = returns.mean()
cov = returns.cov()
weights = np.array([0.074, 0.926])  

stats, annual_mu, annual_cov = optimize.annualized_stats(
    returns,
    interval,
)

print("\nAnnualized ETF statistics:")
print(stats.T.round(4))

current_gm = optimize.geometric_mean_portfolio(weights, mu, cov)
current_sharpe = optimize.sharpe_ratio(weights, mu, cov, risk_free)

print(f'\nCurrent {interval} GM:', round(current_gm, 6))
print("Approx. annualized GM:", round((1 + current_gm) ** anualization_factor - 1, 6))
print(f'Current {interval} Sharpe:', round(current_sharpe, 6))
print("Approx. annualized Sharpe:", round(current_sharpe * np.sqrt(anualization_factor), 6))

[*********************100%***********************]  2 of 2 completed


Annualized ETF statistics:
Ticker             IUSN.DE  VWCE.DE
historical_return   0.0972   0.1273
arithmetic_return   0.1084   0.1295
volatility          0.1748   0.1349

Current 1mo GM: 0.009889
Approx. annualized GM: 0.125342
Current 1mo Sharpe: 0.22821
Approx. annualized Sharpe: 0.790542


Optimization

In [89]:
# Settings for optimization
tickers = ["VWCE.DE", "IUSN.DE"] # current portfolio
interval = "1mo"  # "1d", "1wk", or "1mo"
annual_risk_free = 0.02
anualization_factor = conversion_to_annual[interval]
risk_free = (1 + annual_risk_free) ** (1 / anualization_factor) - 1

In [91]:
# Optimize

data = yf.download(
    tickers,
    interval=interval,
    auto_adjust=True,
    
    period="max",
)
timeseries = data["Close"]
returns = timeseries.pct_change()
print('min date', returns.index.min())
print('max date', returns.index.max())

mu = returns.mean().to_numpy()
cov = returns.cov().to_numpy()

stats, annual_mu, annual_cov = optimize.annualized_stats(
    returns,
    interval,
)
print("\nAnnualized ETF statistics:")
print(stats.T.round(4))

sharpe = optimize.maximize_sharpe_ratio(mu, cov, risk_free)
gm = optimize.maximize_geometric_mean(mu, cov)

weights = pd.DataFrame(
    {
        "ETF": returns.columns,
        "Sharpe_weight": np.round(sharpe["weights"], 6),
        "GM_weight": np.round(gm["weights"], 6),
    }
).set_index("ETF")
print("\nPortfolio weights:")
print(weights.round(4))
print(f"\nOptimal {interval} Sharpe:", round(sharpe["optimal_sharpe"], 6))
print("Approx. annualized Sharpe:", round(sharpe["optimal_sharpe"] * np.sqrt(anualization_factor), 6))
print(f"Optimal {interval} GM:", round(gm["optimal_gm"], 6))
print("Approx. annualized GM:", round((1 + gm["optimal_gm"]) ** anualization_factor - 1, 6))


[*********************100%***********************]  2 of 2 completed

min date 2018-04-01 00:00:00
max date 2026-08-01 00:00:00

Annualized ETF statistics:
Ticker             IUSN.DE  VWCE.DE
historical_return   0.0972   0.1273
arithmetic_return   0.1084   0.1295
volatility          0.1748   0.1349

Portfolio weights:
         Sharpe_weight  GM_weight
ETF                              
IUSN.DE            0.0        0.0
VWCE.DE            1.0        1.0

Optimal 1mo Sharpe: 0.234727
Approx. annualized Sharpe: 0.813119
Optimal 1mo GM: 0.01004
Approx. annualized GM: 0.12736
